# Lab 9: Topic Modeling with LDA

This notebook performs topic modeling on the `npr.csv` dataset using **Gensim** and **LDA**.

Steps:
1. Import libraries
2. Load data
3. Preprocess text
4. Create document-term matrix
5. Run LDA
6. Interpret results
   

In [1]:
import sys
!{sys.executable} -m pip install pandas nltk gensim

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels


In [2]:
import pandas as pd
import nltk
from nltk.corpus import stopwords   
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from gensim import corpora
from gensim.models import LdaModel

In [3]:
# Download NLTK resources

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/f8b31f9b-0f8b-4587-aa37-
[nltk_data]     fee3df21bbc1/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /home/f8b31f9b-0f8b-4587-aa37-
[nltk_data]     fee3df21bbc1/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/f8b31f9b-0f8b-4587-aa37-
[nltk_data]     fee3df21bbc1/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## Load the Data

In [4]:
# Load dataset

df = pd.read_csv('npr.csv')
documents = df['Article'].astype(str).tolist()

print("Total documents:", len(documents))
print("\nFirst article:\n")
print(documents[0])

Total documents: 11992

First article:

In the Washington of 2016, even when the policy can be bipartisan, the politics cannot. And in that sense, this year shows little sign of ending on Dec. 31. When President Obama moved to sanction Russia over its alleged interference in the U. S. election just concluded, some Republicans who had long called for similar or more severe measures could scarcely bring themselves to approve. House Speaker Paul Ryan called the Obama measures ”appropriate” but also ”overdue” and ”a prime example of this administration’s ineffective foreign policy that has left America weaker in the eyes of the world.” Other GOP leaders sounded much the same theme. ”[We have] been urging President Obama for years to take strong action to deter Russia’s worldwide aggression, including its   operations,” wrote Rep. Devin Nunes,  . chairman of the House Intelligence Committee. ”Now with just a few weeks left in office, the president has suddenly decided that some stronger mea

## Preprocess the Data

In [5]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     /home/f8b31f9b-0f8b-4587-aa37-
[nltk_data]     fee3df21bbc1/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/f8b31f9b-0f8b-4587-aa37-
[nltk_data]     fee3df21bbc1/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/f8b31f9b-0f8b-4587-aa37-
[nltk_data]     fee3df21bbc1/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/f8b31f9b-0f8b-4587-aa37-
[nltk_data]     fee3df21bbc1/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [6]:
# Create stopwords set and lemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())                    # convert to lowercase and tokenize
    tokens = [token for token in tokens if token.isalnum()] # remove punctuation/non-alphanumeric
    tokens = [token for token in tokens if token not in stop_words]  # remove stopwords
    tokens = [lemmatizer.lemmatize(token) for token in tokens]       # lemmatization
    return tokens

preprocessed_documents = [preprocess_text(doc) for doc in documents]

print("First preprocessed article:\n")
print(preprocessed_documents[0])

First preprocessed article:

['washington', '2016', 'even', 'policy', 'bipartisan', 'politics', 'sense', 'year', 'show', 'little', 'sign', 'ending', 'president', 'obama', 'moved', 'sanction', 'russia', 'alleged', 'interference', 'election', 'concluded', 'republican', 'long', 'called', 'similar', 'severe', 'measure', 'could', 'scarcely', 'bring', 'approve', 'house', 'speaker', 'paul', 'ryan', 'called', 'obama', 'measure', 'appropriate', 'also', 'overdue', 'prime', 'example', 'administration', 'ineffective', 'foreign', 'policy', 'left', 'america', 'weaker', 'eye', 'gop', 'leader', 'sounded', 'much', 'theme', 'urging', 'president', 'obama', 'year', 'take', 'strong', 'action', 'deter', 'russia', 'worldwide', 'aggression', 'including', 'operation', 'wrote', 'devin', 'nunes', 'chairman', 'house', 'intelligence', 'committee', 'week', 'left', 'office', 'president', 'suddenly', 'decided', 'stronger', 'measure', 'indeed', 'appearing', 'cnn', 'frequent', 'obama', 'critic', 'trent', 'frank', 'call

## Create Document-Term Matrix

In [7]:
# Create dictionary
dictionary = corpora.Dictionary(preprocessed_documents)

# Filter words that appear in less than 15 documents
# and words that appear in more than 50% of documents
dictionary.filter_extremes(no_below=15, no_above=0.5)

# Create corpus
corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]

print("Number of unique tokens after filtering:", len(dictionary))

Number of unique tokens after filtering: 15974


## Run LDA Model

In [ ]:
# Train the LDA model

lda_model = LdaModel(
    corpus=corpus,
    num_topics=5,
    id2word=dictionary,
    passes=15,
    random_state=42
)

## Assign Dominant Topic to Each Article

In [ ]:
# Determine dominant topic for each article

article_labels = []

for doc in preprocessed_documents:
    bow = dictionary.doc2bow(doc)
    topics = lda_model.get_document_topics(bow)
    
    if topics:
        dominant_topic = max(topics, key=lambda x: x[1])[0]
    else:
        dominant_topic = -1
        
    article_labels.append(dominant_topic)

df_result = pd.DataFrame({
    "Article": documents,
    "Topic": article_labels
})

print("Table with Articles and Topic:")
df_result

## Show Top Terms for Each Topic

In [ ]:
# Print top 10 words for each topic

for topic_id in range(lda_model.num_topics):
    print(f"Top terms for Topic #{topic_id}:")
    top_terms = lda_model.show_topic(topic_id, topn=10)
    print([term[0] for term in top_terms])
    print()

## Show Top Terms with Weights

In [ ]:
print("Top Terms for Each Topic:")

for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}:")
    terms = [term.strip() for term in topic.split("+")]
    for term in terms:
        weight, word = term.split("*")
        print(f"- {word.strip()} (weight: {weight.strip()})")
    print()

## Add Topic Names

In [ ]:
# Topic labels based on interpretation from the lab

topic_names = {
    0: "Education",
    1: "Personal",
    2: "World Politics",
    3: "Health",
    4: "US Election"
}

df_result["Topic_Name"] = df_result["Topic"].map(topic_names)

df_result.head(10)

In [ ]:
# Count how many articles belong to each topic

print("Number of articles in each topic:")
print(df_result["Topic_Name"].value_counts())

## Conclusion

The LDA model grouped the NPR articles into 5 main topics based on word distributions. 
After analyzing the top keywords in each topic, the topics were interpreted as:

- Education: related to schools, students, and universities
- Personal: focused on life, stories, and personal experiences
- World Politics: includes government, war, and global issues
- Health: related to medical care, patients, and health topics
- US Election: includes political figures, campaigns, and elections

This shows that topic modeling can effectively identify hidden themes in large text datasets.